<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module3_Labs/Lab10.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 10 — From Energy Estimation to VQE Optimization
**Quantum Optimization and Simulation — VQE Laboratory Series**

Lab 9 evaluated the energy of one two-qubit trial state. This lab uses the **same circuit and the same Hamiltonian**, but lets COBYLA vary the angle automatically to search for a lower energy.

After the ideal simulation, we make the experiment more realistic with a fake IBM noise model. Real IBM hardware is included as an **optional** final section.

**Suggested use:** guided lab. Most code is supplied.

> The goal is the VQE loop: **prepare → measure → calculate energy → adjust \(	heta\) → repeat**.


## Learning objectives
- Reuse the Lab 9 two-qubit circuit as a variational ansatz.
- Use COBYLA to minimize the measured energy.
- Count how many measurement circuits VQE executes.
- Compare ideal and noisy simulation.
- Explain why more shots reduce random variation but do not remove hardware-noise bias.
- Optionally evaluate the optimized circuit on real IBM Quantum hardware.


In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install pylatexenc matplotlib
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"
%pip -q install "qiskit-ibm-runtime~=0.47"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

SEED = 123
SHOTS = 4096

# Same reduced H2 Hamiltonian coefficients used in Lab 3.
c0 = -1.0523732458
c1 = +0.3979374248
c2 = -0.3979374248
c3 = -0.0112801043
c4 = +0.1809311998

CIRCUIT_EXECUTIONS = 0

def reset_execution_counter():
    global CIRCUIT_EXECUTIONS
    CIRCUIT_EXECUTIONS = 0

def run_counts(qc, shots=SHOTS, noise_model=None, seed=SEED, optimization_level=1):
    global CIRCUIT_EXECUTIONS

    backend = AerSimulator(noise_model=noise_model)

    kwargs = {}
    if noise_model is not None:
        kwargs["basis_gates"] = noise_model.basis_gates

    tqc = transpile(
        qc,
        backend,
        optimization_level=optimization_level,
        seed_transpiler=SEED,
        #**kwargs,
    )
    result = backend.run(tqc, shots=shots, seed_simulator=seed).result()

    CIRCUIT_EXECUTIONS += 1
    return result.get_counts()


## Part A — Use the same circuit as Lab 9

The trial state is

```text
q0: ─X──────■──
            │
q1: ─Ry(2θ)─■──
```

with the CNOT controlled by $q_1$ and targeting $q_0$.

Changing $\theta$ changes the mixture of the two configurations. In Lab 9 you chose $\theta$. Here COBYLA will choose it.


In [ ]:
def ansatz(theta):
    qc = QuantumCircuit(2)
    qc.x(0)
    qc.ry(2 * float(theta), 1)
    qc.cx(1, 0)
    return qc

display(ansatz(0.45).draw("mpl"))


## Part B — Reuse the Lab 9 measurement method

In [ ]:
def parity_expectation(counts, qubits):
    total = sum(counts.values())
    value = 0.0

    for bits, count in counts.items():
        bits = bits.replace(" ", "")
        eigenvalue = 1
        for q in qubits:
            bit = int(bits[-1-q])
            eigenvalue *= (1 if bit == 0 else -1)
        value += eigenvalue * count / total

    return value


def measure_pauli_pair(base_circuit, basis, shots=SHOTS,
                       noise_model=None, seed=SEED, optimization_level=1):
    qc = base_circuit.copy()

    if basis == "X":
        qc.h([0, 1])
    elif basis == "Y":
        qc.sdg([0, 1])
        qc.h([0, 1])

    qc.measure_all()

    return run_counts(
        qc,
        shots=shots,
        noise_model=noise_model,
        seed=seed,
        optimization_level=optimization_level,
    )


def measured_energy(theta, shots=SHOTS, noise_model=None,
                    seed=SEED, optimization_level=1):
    qc = ansatz(theta)

    # Three measurement settings, exactly as in Lab 3.
    z_counts = measure_pauli_pair(
        qc, "Z", shots, noise_model, seed, optimization_level
    )
    x_counts = measure_pauli_pair(
        qc, "X", shots, noise_model, seed + 1, optimization_level
    )
    y_counts = measure_pauli_pair(
        qc, "Y", shots, noise_model, seed + 2, optimization_level
    )

    z0 = parity_expectation(z_counts, [0])
    z1 = parity_expectation(z_counts, [1])
    z0z1 = parity_expectation(z_counts, [0, 1])
    x0x1 = parity_expectation(x_counts, [0, 1])
    y0y1 = parity_expectation(y_counts, [0, 1])

    return (
        c0
        + c1*z0
        + c2*z1
        + c3*z0z1
        + c4*(x0x1 + y0y1)
    )


### Quick check

Each energy estimate uses **three** measurement circuits: one each for the Z, X, and Y measurement settings.


In [ ]:
reset_execution_counter()

test_energy = measured_energy(0.45)

print("Energy at theta=0.45:", test_energy, "Hartree")
print("Quantum measurement circuits executed:", CIRCUIT_EXECUTIONS)


## Part C — Let COBYLA optimize the angle

COBYLA is a classical optimizer. It proposes an angle, asks the quantum experiment for an energy, then proposes another angle.

Because each energy evaluation uses three measurement circuits, the execution count grows quickly.


In [ ]:
reset_execution_counter()
ideal_history = []

def ideal_objective(theta_array):
    theta_value = float(np.atleast_1d(theta_array)[0])
    value = measured_energy(
        theta_value,
        shots=SHOTS,
        noise_model=None,
        seed=SEED + len(ideal_history) * 10,
    )
    ideal_history.append((theta_value, value))
    return value

result_ideal = minimize(
    ideal_objective,
    x0=np.array([0.0]),
    method="COBYLA",
    options={"maxiter": 35, "rhobeg": 0.2, "tol": 1e-4},
)

theta_ideal = float(result_ideal.x[0])
energy_ideal = float(result_ideal.fun)

print("Optimal theta:", theta_ideal)
print("Ideal Aer VQE energy:", energy_ideal, "Hartree")
print("Energy evaluations:", len(ideal_history))
print("Quantum measurement circuits executed:", CIRCUIT_EXECUTIONS)

history = np.array(ideal_history)
plt.plot(history[:, 1], marker=".")
plt.xlabel("Energy evaluation")
plt.ylabel("Energy (Hartree)")
plt.title("COBYLA convergence — ideal AerSimulator")
plt.grid(True)
plt.show()


### YOUR TURN 1

If COBYLA evaluates the energy 20 times, approximately how many measurement circuits are executed in this lab?

<details>
<summary><b>Suggested answer</b></summary>

Three measurement settings are used for each energy evaluation, so about \(20\times3=60\) circuits are executed.

</details>


## Part D — Add a fake IBM-device noise model

A fake backend stores a snapshot of realistic device properties such as gate errors and readout errors. We use those properties in `AerSimulator`.

The fake backend is still a simulator: no real quantum computer is used in this section.


In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeVigoV2

fake_backend = FakeVigoV2()
fake_noise = NoiseModel.from_backend(fake_backend)

reset_execution_counter()
noisy_history = []

def noisy_objective(theta_array):
    theta_value = float(np.atleast_1d(theta_array)[0])
    value = measured_energy(
        theta_value,
        shots=SHOTS,
        noise_model=fake_noise,
        seed=SEED + len(noisy_history) * 10,
        optimization_level=1,
    )
    noisy_history.append((theta_value, value))
    return value

result_noisy = minimize(
    noisy_objective,
    x0=np.array([theta_ideal]),
    method="COBYLA",
    options={"maxiter": 25, "rhobeg": 0.15, "tol": 1e-4},
)

theta_noisy = float(result_noisy.x[0])
energy_noisy = float(result_noisy.fun)

print("Noisy optimal theta:", theta_noisy)
print("Noisy VQE energy:", energy_noisy, "Hartree")
print("Energy evaluations:", len(noisy_history))
print("Quantum measurement circuits executed:", CIRCUIT_EXECUTIONS)


## Part E — Why doesn't the noisy result reach the ideal energy?

Two effects are mixed together:

$
\text{measured energy}
\approx
\text{ideal energy}
+\text{hardware-noise bias}
+\text{sampling fluctuation}.
$

Increasing the number of shots reduces the **sampling fluctuation**. It does **not** remove systematic gate, decoherence, or readout errors.

So we should expect more shots to make repeated answers more stable, but not necessarily make their average equal to the ideal answer.


In [ ]:
# Keep theta fixed and repeat the noisy energy measurement.
# This isolates the effect of the number of shots from the optimizer.

shot_values = [500, 2_000, 8_000]
repetitions = 5

means = []
stds = []

reset_execution_counter()

for shots in shot_values:
    values = []

    for rep in range(repetitions):
        value = measured_energy(
            theta_ideal,
            shots=shots,
            noise_model=fake_noise,
            seed=SEED + 100*rep + shots,
            optimization_level=1,
        )
        values.append(value)

    means.append(np.mean(values))
    stds.append(np.std(values, ddof=1))

    print(
        f"shots={shots:5d} : "
        f"mean={np.mean(values): .6f} Ha, "
        f"std={np.std(values, ddof=1):.6f}"
    )

print("Quantum measurement circuits executed for shot study:", CIRCUIT_EXECUTIONS)

plt.errorbar(shot_values, means, yerr=stds, marker="o", capsize=4)
plt.axhline(energy_ideal, linestyle="--", label="Ideal Aer VQE")
plt.xscale("log")
plt.xlabel("Shots per measurement circuit")
plt.ylabel("Energy (Hartree)")
plt.title("More shots reduce variation, not all noise bias")
plt.legend()
plt.grid(True)
plt.show()


### A second practical improvement: compile the circuit more carefully

We can also ask Qiskit's transpiler to work harder to reduce circuit depth and unnecessary gates. This is **error suppression**, not full error correction.

Try the same noisy energy at the ideal angle with optimization levels 1 and 3.


In [ ]:
reset_execution_counter()

e_level1 = measured_energy(
    theta_ideal,
    shots=8_000,
    noise_model=fake_noise,
    seed=SEED,
    optimization_level=1,
)

e_level3 = measured_energy(
    theta_ideal,
    shots=8_000,
    noise_model=fake_noise,
    seed=SEED,
    optimization_level=3,
)

print("Noisy energy, transpiler level 1:", e_level1)
print("Noisy energy, transpiler level 3:", e_level3)
print("Ideal Aer VQE energy:             ", energy_ideal)
print("Quantum measurement circuits executed:", CIRCUIT_EXECUTIONS)


### YOUR TURN 2

Did more shots remove all of the difference between the noisy and ideal answers? Why or why not?

<details>
<summary><b>Suggested answer</b></summary>

Usually no. More shots reduce random sampling variation, but the fake device also includes systematic gate and readout errors. Those errors can leave a persistent bias.

</details>


## Optional — Run the optimized circuit on real IBM Quantum hardware

This section is optional because it requires an IBM Quantum account, available hardware, queue time, and current Runtime access.

To keep QPU usage small, we **do not optimize on hardware**. We take the angle found by the simulator and run only the three measurement circuits needed to estimate its energy.

> IBM Runtime APIs can evolve. If this optional cell needs a small update in a future course offering, follow the current IBM Quantum Runtime documentation.


In [ ]:
# OPTIONAL REAL-HARDWARE TEMPLATE — do not run unless you have IBM Quantum access.

# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
#
# service = QiskitRuntimeService()
# real_backend = service.least_busy(
#     operational=True,
#     simulator=False,
#     min_num_qubits=2,
# )
#
# def make_measurement_circuit(theta_value, basis):
#     qc = ansatz(theta_value)
#     if basis == "X":
#         qc.h([0, 1])
#     elif basis == "Y":
#         qc.sdg([0, 1])
#         qc.h([0, 1])
#     qc.measure_all()
#     return qc
#
# circuits = [
#     make_measurement_circuit(theta_ideal, "Z"),
#     make_measurement_circuit(theta_ideal, "X"),
#     make_measurement_circuit(theta_ideal, "Y"),
# ]
#
# pm = generate_preset_pass_manager(
#     backend=real_backend,
#     optimization_level=3,
# )
# isa_circuits = [pm.run(c) for c in circuits]
#
# sampler = SamplerV2(mode=real_backend)
# job = sampler.run(isa_circuits, shots=4096)
# pub_results = job.result()
#
# counts_z = pub_results[0].data.meas.get_counts()
# counts_x = pub_results[1].data.meas.get_counts()
# counts_y = pub_results[2].data.meas.get_counts()
#
# print("Real hardware measurement circuits executed: 3")
# print("Backend:", real_backend.name)
#
# # Reuse parity_expectation(...) to reconstruct the energy.


## Final reflection

1. What does the quantum part of VQE provide to COBYLA?
2. Why does one energy evaluation require three measurement circuits here?
3. What does increasing shots improve?
4. What kinds of error are not fixed merely by increasing shots?
5. Why is it sensible to optimize first on a simulator before using real hardware?
